In [ ]:
import json
import random
from pathlib import Path
from typing import Dict, List, Optional, Tuple
from collections import defaultdict
import importlib.util

# Set random seed for reproducibility
random.seed(42)

# Task name to folder name mapping
TASK_TO_FOLDER = {
    "Concept Explanation": "concept_explanation",
    "Plan & Design": "plan_and_design",
    "Analyze & Critique": "analyze_and_critique",
    "Revise": "revise"
}

ALL_CONTEXT_MERGE_DIR = Path(".").resolve()
if not (ALL_CONTEXT_MERGE_DIR / "query_rules.py").exists():
    ALL_CONTEXT_MERGE_DIR = Path("data_pipeline/context_merge").resolve()
FRAMEWORK_DIR = ALL_CONTEXT_MERGE_DIR.parent

# Load user profiles
user_profile_path = ALL_CONTEXT_MERGE_DIR / "user_profile.py"
if user_profile_path.exists():
    spec = importlib.util.spec_from_file_location("user_profile", user_profile_path)
    user_profile_module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(user_profile_module)
    ALL_PERSONAS = user_profile_module.ALL_PERSONAS
else:
    ALL_PERSONAS = {}

# Load subject to directory_index mapping for cross-session
cross_session_topics_research_path = FRAMEWORK_DIR / "cross_session_generation" / "cross_session_topics_research.json"
cross_session_topics_tutoring_path = FRAMEWORK_DIR / "cross_session_generation" / "cross_session_topics_tutoring.json"

SUBJECT_TO_DIRECTORIES_BY_DOMAIN = {}
if cross_session_topics_research_path.exists():
    with open(cross_session_topics_research_path, 'r', encoding='utf-8') as f:
        research_topics = json.load(f)
        SUBJECT_TO_DIRECTORIES_BY_DOMAIN["research"] = {}
        for subject, topics in research_topics.items():
            SUBJECT_TO_DIRECTORIES_BY_DOMAIN["research"][subject] = [t["directory_index"] for t in topics]

if cross_session_topics_tutoring_path.exists():
    with open(cross_session_topics_tutoring_path, 'r', encoding='utf-8') as f:
        tutoring_topics = json.load(f)
        SUBJECT_TO_DIRECTORIES_BY_DOMAIN["tutoring"] = {}
        for subject, topics in tutoring_topics.items():
            SUBJECT_TO_DIRECTORIES_BY_DOMAIN["tutoring"][subject] = [t["directory_index"] for t in topics]

In [20]:
def load_events_with_queries(output_dir: Path, domain: str, directory_index: str) -> List[Dict]:
    """Load events with queries from output directory."""
    events_file = output_dir / domain / directory_index / "events.json"
    if not events_file.exists():
        return []
    with open(events_file, 'r', encoding='utf-8') as f:
        events = json.load(f)
    return events


def load_artifact_history(mix_dir: Path, domain: str, directory_index: str) -> Dict:
    """Load artifact history from mix directory."""
    artifact_store_file = mix_dir / domain / directory_index / "artifacts" / "artifact_store.json"
    artifact_history = {}
    if artifact_store_file.exists():
        with open(artifact_store_file, 'r', encoding='utf-8') as f:
            artifact_store = json.load(f)
            if "history" in artifact_store:
                artifact_history = artifact_store["history"]
    return artifact_history


def get_latest_artifact_version(artifact_history: Dict, artifact_label: str, 
                                current_event_idx: int, events: List[Dict]) -> Optional[Dict]:
    """
    Get the latest version of an artifact before or at the current event.
    Returns the artifact dict with the highest time_index <= current_event's time_index.
    """
    if artifact_label not in artifact_history:
        return None
    
    # Create event_id to index mapping
    event_id_to_idx = {event["event_id"]: i for i, event in enumerate(events)}
    current_time_index = events[current_event_idx]["time_index"]
    
    # Find the artifact version with the highest time_index <= current_time_index
    latest_version = None
    latest_time_index = -1
    
    for artifact_version in artifact_history[artifact_label]:
        if "event_id" in artifact_version and "time_index" in artifact_version:
            event_id = artifact_version["event_id"]
            time_index = artifact_version["time_index"]
            
            if event_id in event_id_to_idx:
                event_idx = event_id_to_idx[event_id]
                if event_idx <= current_event_idx and time_index > latest_time_index:
                    latest_version = artifact_version
                    latest_time_index = time_index
    
    return latest_version


In [33]:
def get_user_profile(domain: str, subject: str) -> str:
    """Get a random user profile for the given domain and subject."""
    if domain not in ALL_PERSONAS:
        return ""
    if subject not in ALL_PERSONAS[domain]:
        return ""
    personas = ALL_PERSONAS[domain][subject]
    if not personas:
        return ""
    return random.choice(personas)


def load_cross_session_cases(cross_session_dir: Path, domain: str, subject: str, 
                              task: str, num_cases: int = 2) -> List[Dict]:
    """
    Load random cross-session cases for the given domain, subject, and task.
    Returns a list of dicts, each containing 'regime' and 'task_summary'.
    """
    # Map task name to folder name
    task_folder = TASK_TO_FOLDER.get(task, task.lower().replace(" ", "_").replace("&", "and"))
    
    # Get all directory_index for this subject in this domain
    if domain not in SUBJECT_TO_DIRECTORIES_BY_DOMAIN:
        return []
    if subject not in SUBJECT_TO_DIRECTORIES_BY_DOMAIN[domain]:
        return []
    
    directory_indices = SUBJECT_TO_DIRECTORIES_BY_DOMAIN[domain][subject]
    
    # Collect all valid cross-session files
    all_summary_files = []
    for directory_index in directory_indices:
        task_dir = cross_session_dir / domain / directory_index / task_folder
        if not task_dir.exists():
            continue
        
        # Find all cross_session_r*.json files (excluding interactions and stats files)
        summary_files = list(task_dir.glob("cross_session_r*.json"))
        # Filter out interactions and stats files
        summary_files = [f for f in summary_files if "_interactions" not in f.name and "_stats" not in f.name]
        all_summary_files.extend(summary_files)
    
    if not all_summary_files:
        return []
    
    # Randomly select num_cases files (or all if fewer available)
    selected_files = random.sample(all_summary_files, min(num_cases, len(all_summary_files)))
    
    cases = []
    for selected_file in selected_files:
        try:
            with open(selected_file, 'r', encoding='utf-8') as f:
                data = json.load(f)
                summary = data.get("summary", "")
                regime_id = data.get("metadata", {}).get("regime_id", None)
                if summary and regime_id is not None:
                    cases.append({
                        "regime": regime_id,
                        "task_summary": summary
                    })
        except Exception as e:
            print(f"  Warning: Error loading cross-session case from {selected_file}: {e}")
            continue
    
    return cases


def build_context_prompt(events: List[Dict], event_idx: int, query_info: Dict, 
                        artifact_history: Dict, user_profile: str = "", 
                        cross_session_cases: List[Dict] = None) -> str:
    """
    Build context for a query (without the user query itself).
    
    Includes:
    1. User Profile (if provided)
    2. Retrieved History Task Interaction Summary (cross-session cases)
    3. Overview of the past 5 events
    4. Latest versions of essential artifacts
    """
    context_parts = []
    
    # Add user profile if provided, otherwise add placeholder
    if user_profile:
        context_parts.append(f"""## User Profile

{user_profile}
""")
    else:
        context_parts.append("""## User Profile

No user profile available.
""")
    
    # Add cross-session summaries if provided
    if cross_session_cases and len(cross_session_cases) > 0:
        cross_session_text = "## Summary of Historical Session Interactions\n\n"
        for i, case in enumerate(cross_session_cases, 1):
            task_summary = case.get("task_summary", "")
            regime = case.get("regime", "")
            if task_summary:
                cross_session_text += f"**Historical Session {i}:**\n{task_summary}\n\n"
        context_parts.append(cross_session_text)
    else:
        context_parts.append("""## Summary of Historical Session Interactions

No retrieved historical session interaction summary available.
""")
    
    # Get past 5 events (or as many as available)
    start_idx = max(0, event_idx - 5)
    past_events = events[start_idx:event_idx]
    
    # Build event overview
    event_overview = []
    for i, event in enumerate(past_events):
        event_num = start_idx + i + 1
        event_summary = f"Event {event_num} ({event.get('event_type', 'unknown')}):\n{event.get('description', '')}\n"
        event_overview.append(event_summary)
    
    # Get latest versions of essential artifacts
    essential_artifacts = query_info.get("essential", [])
    artifact_contents = {}
    
    for artifact_label in essential_artifacts:
        latest_version = get_latest_artifact_version(
            artifact_history, artifact_label, event_idx, events
        )
        if latest_version and "text" in latest_version:
            artifact_contents[artifact_label] = latest_version["text"]
    
    # Build project context
    project_context = "## Project Context\n\n"
    
    # Add Recent Events section
    project_context += "### Recent Events\n\n"
    if event_overview:
        project_context += f"{chr(10).join(event_overview)}\n"
    else:
        project_context += "No recent events available.\n\n"
    
    # Add Relevant Artifacts section
    project_context += "### Relevant Artifacts\n"
    if artifact_contents:
        for artifact_label, artifact_text in artifact_contents.items():
            project_context += f"""
======================== **{artifact_label.replace('_', ' ').title()}** starts here ========================
{artifact_text}
======================== **{artifact_label.replace('_', ' ').title()}** ends here =========================
"""
    else:
        project_context += "\nNo relevant artifacts available.\n"
    
    context_parts.append(project_context)
    
    return "\n".join(context_parts)


In [34]:
def get_all_queries_from_timeline(events: List[Dict]) -> List[Dict]:
    """
    Get all queries from a timeline (no sampling, no filtering).
    Returns a list of dicts, each containing event_idx, event, and query_info.
    """
    all_queries = []
    for i, event in enumerate(events):
        queries = event.get("query", [])
        for query_info in queries:
            all_queries.append({
                "event_idx": i,
                "event": event,
                "query_info": query_info
            })
    return all_queries


In [35]:
def process_all_timelines(output_dir: Path, mix_dir: Path, cross_session_dir: Path,
                          domains: List[str]) -> List[Dict]:
    """
    Process all timelines and build context for all queries.
    
    Returns:
        List of context dicts, each containing:
        - domain, subject, topic
        - task, directory_index, event_id
        - query, target
        - essential_targets (dict)
        - past_5_events (list)
        - userprofile
        - cross_session (list of dicts with regime and task_summary)
        - full_context
    """
    output_dir = Path(output_dir)
    mix_dir = Path(mix_dir)
    cross_session_dir = Path(cross_session_dir)
    
    all_contexts = []
    
    for domain in domains:
        domain_output_dir = output_dir / domain
        if not domain_output_dir.exists():
            print(f"Warning: Domain directory {domain_output_dir} does not exist")
            continue
        
        # Get all numbered subdirectories
        subdirs = [d for d in domain_output_dir.iterdir() if d.is_dir() and d.name.isdigit()]
        subdirs.sort(key=lambda x: int(x.name))
        
        print(f"\nProcessing {domain} domain: {len(subdirs)} topics")
        
        for subdir in subdirs:
            directory_index = subdir.name
            
            try:
                # Load events with queries
                events = load_events_with_queries(output_dir, domain, directory_index)
                if not events:
                    continue
                
                # Load artifact history
                artifact_history = load_artifact_history(mix_dir, domain, directory_index)
                
                # Get subject from first event (should be consistent across events)
                subject = events[0].get("subject", "") if events else ""
                
                # Get all queries from this timeline
                all_queries = get_all_queries_from_timeline(events)
                
                # Build context for each query
                for sample in all_queries:
                    event_idx = sample["event_idx"]
                    event = sample["event"]
                    query_info = sample["query_info"]
                    
                    # Get user profile
                    user_profile = get_user_profile(domain, subject)
                    
                    # Get cross-session cases (2 random cases from same subject)
                    cross_session_cases = load_cross_session_cases(
                        cross_session_dir, domain, subject, query_info["task"], num_cases=2
                    )
                    
                    # Get past 5 events
                    start_idx = max(0, event_idx - 5)
                    past_events = events[start_idx:event_idx]
                    past_5_events = [
                        {
                            "event_id": e.get("event_id", ""),
                            "description": e.get("description", "")
                        }
                        for e in past_events
                    ]
                    
                    # Get essential targets (latest versions of essential artifacts)
                    essential_artifacts = query_info.get("essential", [])
                    essential_targets = {}
                    for artifact_label in essential_artifacts:
                        latest_version = get_latest_artifact_version(
                            artifact_history, artifact_label, event_idx, events
                        )
                        if latest_version and "text" in latest_version:
                            essential_targets[artifact_label] = latest_version["text"]
                    
                    # Build context prompt
                    full_context = build_context_prompt(
                        events, event_idx, query_info, artifact_history,
                        user_profile=user_profile,
                        cross_session_cases=cross_session_cases
                    )
                    
                    context_dict = {
                        "domain": domain,
                        "subject": subject,
                        "topic": event.get("topic", ""),
                        "task": query_info["task"],
                        "directory_index": directory_index,
                        "event_id": event.get("event_id", ""),
                        "query": query_info["query"],
                        "target": query_info["target"],
                        "essential_artifacts": essential_targets,
                        "past_5_events": past_5_events,
                        "user_profile": user_profile,
                        "cross_session": cross_session_cases,
                        "full_context": full_context
                    }
                    
                    all_contexts.append(context_dict)
                    
            except Exception as e:
                print(f"  Error processing {domain}/{directory_index}: {e}")
                import traceback
                traceback.print_exc()
                continue
    
    return all_contexts


In [ ]:
# Main execution
ALL_CONTEXT_MERGE_DIR = Path(".").resolve()
if not (ALL_CONTEXT_MERGE_DIR / "query_rules.py").exists():
    ALL_CONTEXT_MERGE_DIR = Path("data_pipeline/context_merge").resolve()
FRAMEWORK_DIR = ALL_CONTEXT_MERGE_DIR.parent

output_dir = ALL_CONTEXT_MERGE_DIR / "output"
mix_dir = FRAMEWORK_DIR / "scenario_assembly"
cross_session_dir = FRAMEWORK_DIR / "cross_session_generation" / "cross_session_context"

# Process all timelines
contexts = process_all_timelines(
    output_dir, mix_dir, cross_session_dir,
    domains=["research", "tutoring"]
)

print(f"\n=== Summary ===")
print(f"Total contexts generated: {len(contexts)}")

# Count by domain and task
by_domain = defaultdict(int)
by_task = defaultdict(int)
for ctx in contexts:
    by_domain[ctx["domain"]] += 1
    by_task[ctx["task"]] += 1

print(f"\nBy domain:")
for domain, count in by_domain.items():
    print(f"  {domain}: {count} contexts")

print(f"\nBy task:")
for task, count in by_task.items():
    print(f"  {task}: {count} contexts")

# Save contexts
output_file = ALL_CONTEXT_MERGE_DIR / "all_contexts.json"
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(contexts, f, indent=4, ensure_ascii=False)

print(f"\nSaved contexts to: {output_file}")